<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-14/notebooks/ClimatePipeline/07_ClimateMunicipalAudit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClimateMunicipalAudit

Audita la capa diaria municipal de precipitación sin modificarla ni imputar ausencias.

## Preguntas de esta compuerta

- ¿Qué municipios y periodos tienen cobertura temporal suficiente para construir indicadores?
- ¿Dónde están los días con cobertura insuficiente de estaciones?
- ¿Cuánto cambian los resultados al usar media en lugar de mediana?
- ¿Qué municipios multiestación requieren revisión antes de aprobar la regla v1?

Los umbrales de lluvia son diagnósticos de sensibilidad. Esta auditoría no aprueba un umbral, no elimina filas y no habilita todavía el paso 08.

## 1. Preparar el repositorio

La celda actualiza explícitamente `feature/SCRUM-14` para evitar módulos antiguos en Colab.

In [ ]:
from pathlib import Path

import subprocess
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-15'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
ACTUALIZAR_REPOSITORIO = True

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    elif ACTUALIZAR_REPOSITORIO:
        remote_ref = f'refs/remotes/origin/{REPO_REF}'
        subprocess.run(
            [
                'git', 'fetch', '--depth', '1', 'origin',
                f'+refs/heads/{REPO_REF}:{remote_ref}',
            ],
            cwd=REPO_DIR,
            check=True,
        )
        subprocess.run(
            ['git', 'checkout', '-B', REPO_REF, remote_ref],
            cwd=REPO_DIR,
            check=True,
        )

PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
if not PIPELINE_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta del pipeline: {PIPELINE_DIR}')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

print({'in_colab': IN_COLAB, 'repo_ref': REPO_REF, 'repo_dir': str(REPO_DIR)})

## 2. Configuración protegida

Primero ejecute todo con `EJECUTAR_AUDITORIA_MUNICIPAL=False` y confirme el plan. Después cambie únicamente esa bandera a `True`.

In [ ]:
import importlib
import json
import time

import pandas as pd

import ClimateProcessingUtils
import PrecipitationMunicipalAudit

importlib.reload(ClimateProcessingUtils)
importlib.reload(PrecipitationMunicipalAudit)

from ClimateProcessingUtils import (
    ahora_proyecto,
    detectar_commit,
    escribir_json_atomico,
    escribir_parquet_atomico,
    escribir_texto_atomico,
    formatear_duracion,
    slugificar,
)
from DatasetConfig import cargar_configuracion_datasets
from PrecipitationMunicipalAudit import (
    AGGREGATION_VERSION_ESPERADA,
    AUDIT_VERSION,
    auditar_precipitacion_municipal,
)

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

if AUDIT_VERSION != 'auditoria_precipitacion_municipal_v1':
    raise RuntimeError(f'Versión de auditoría inesperada: {AUDIT_VERSION}')

VARIABLE_NOMBRE = 'precipitacion'
DATASET_ID = 's54a-sgyg'
AGREGACION_ENTRADA = 'precipitacion_municipio_dia_2024_2025_v1'
AUDITORIA_NOMBRE = 'cierre_precipitacion_municipal_2024_2025_v1'
UMBRALES_LLUVIA_MM = [0.1, 1.0, 5.0, 10.0, 20.0]

EJECUTAR_AUDITORIA_MUNICIPAL = False
GUARDAR_RESULTADOS = True
SOBRESCRIBIR_AUDITORIA = False

DATASET_CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)
PROCESSED_ROOT = DATASET_CONFIG.processed_root
INPUT_DIR = (
    PROCESSED_ROOT
    / 'clima_municipal'
    / f'variable={VARIABLE_NOMBRE}'
    / f'fuente={DATASET_ID}'
    / f'agregacion={slugificar(AGREGACION_ENTRADA)}'
)
OUTPUT_DIR = (
    PROCESSED_ROOT
    / 'auditorias_clima_municipal'
    / f'variable={VARIABLE_NOMBRE}'
    / f'fuente={DATASET_ID}'
    / f'auditoria={slugificar(AUDITORIA_NOMBRE)}'
)

print({
    'audit_version': AUDIT_VERSION,
    'ejecutar': EJECUTAR_AUDITORIA_MUNICIPAL,
    'guardar': GUARDAR_RESULTADOS,
    'entrada': str(INPUT_DIR),
    'salida': str(OUTPUT_DIR),
})

## 3. Plan y validación de entrada

In [ ]:
def leer_manifest_si_existe(ruta):
    return json.loads(ruta.read_text(encoding='utf-8')) if ruta.exists() else {}


def inspeccionar_plan_auditoria():
    manifest = leer_manifest_si_existe(INPUT_DIR / 'manifest.json')
    particiones = sorted(
        INPUT_DIR.glob(
            'departamento=*/anio=*/mes=*/precipitacion_municipio_dia.parquet'
        )
    )
    return pd.DataFrame([{
        'entrada_estado': manifest.get('estado', 'NO_ENCONTRADA'),
        'entrada_version': manifest.get('aggregation_version'),
        'particiones': len(particiones),
        'filas_manifest': manifest.get('metricas', {}).get('filas_municipio_dia'),
        'municipios_manifest': manifest.get('metricas', {}).get('municipios_objetivo'),
        'salida_existe': OUTPUT_DIR.exists(),
        'salida': str(OUTPUT_DIR),
    }])


plan_auditoria_df = inspeccionar_plan_auditoria()
display(Markdown('### Plan de auditoría municipal'))
display(plan_auditoria_df)

## 4. Funciones de carga, reporte, figuras y persistencia

In [ ]:
NOMBRES_SALIDA = {
    'cobertura_municipios': 'cobertura_municipios.parquet',
    'cobertura_periodos': 'cobertura_periodos.parquet',
    'cobertura_insuficiente': 'cobertura_insuficiente.parquet',
    'multiestacion_dias': 'multiestacion_dias.parquet',
    'resumen_multiestacion': 'resumen_multiestacion.parquet',
    'sensibilidad_anual': 'sensibilidad_media_mediana_anual.parquet',
    'sensibilidad_umbrales': 'sensibilidad_umbrales_lluvia.parquet',
    'grafica_cobertura': 'cobertura_temporal_municipios.html',
    'grafica_sensibilidad': 'sensibilidad_media_mediana.html',
    'reporte': 'AuditoriaMunicipal_precipitacion_2024_2025.md',
    'manifest': 'manifest.json',
}


def cargar_clima_municipal():
    manifest_path = INPUT_DIR / 'manifest.json'
    if not manifest_path.exists():
        raise FileNotFoundError(f'No existe el manifiesto municipal: {manifest_path}')
    manifest = leer_manifest_si_existe(manifest_path)
    if manifest.get('estado') != 'COMPLETA':
        raise RuntimeError('La agregación municipal no está COMPLETA.')
    if manifest.get('aggregation_version') != AGGREGATION_VERSION_ESPERADA:
        raise RuntimeError(
            f'Versión municipal inesperada: {manifest.get("aggregation_version")}'
        )
    archivos = sorted(
        INPUT_DIR.glob(
            'departamento=*/anio=*/mes=*/precipitacion_municipio_dia.parquet'
        )
    )
    if len(archivos) != 48:
        raise RuntimeError(f'Se esperaban 48 particiones y existen {len(archivos)}.')
    tabla = pd.concat([pd.read_parquet(archivo) for archivo in archivos], ignore_index=True)
    filas_manifest = manifest.get('metricas', {}).get('filas_municipio_dia')
    if filas_manifest is not None and len(tabla) != int(filas_manifest):
        raise RuntimeError(f'Filas leídas ({len(tabla):,}) != manifiesto ({filas_manifest:,}).')
    return tabla, manifest, archivos


def tabla_markdown(tabla, limite=None):
    vista = tabla.head(limite) if limite is not None else tabla
    try:
        return vista.to_markdown(index=False)
    except ImportError:
        return '```text\n' + vista.to_string(index=False) + '\n```'


def construir_figuras(resultado):
    try:
        import plotly.express as px
    except ImportError:
        print('Plotly no está disponible; se omiten las figuras.')
        return {}

    cobertura = resultado.cobertura_municipios.loc[
        resultado.cobertura_municipios['estaciones_canonicas_total'].gt(0)
    ].copy()
    figura_cobertura = px.bar(
        cobertura.sort_values('cobertura_sobre_dias_esperados_pct'),
        x='municipio',
        y='cobertura_sobre_dias_esperados_pct',
        color='departamento',
        hover_data=[
            'codigo_municipio', 'dias_validos',
            'dias_con_estacion_esperada', 'clasificacion_cobertura',
        ],
        title='Cobertura temporal de municipios con estación canónica',
        labels={'cobertura_sobre_dias_esperados_pct': 'Cobertura (%)'},
    )
    figura_cobertura.update_layout(height=560, xaxis_tickangle=-70)

    multi = resultado.multiestacion_dias
    figura_sensibilidad = px.scatter(
        multi,
        x='precipitacion_mediana_estaciones_mm',
        y='precipitacion_media_estaciones_mm',
        color='municipio',
        size='estaciones_con_dato',
        hover_data=['fecha', 'codigo_municipio', 'rango_estaciones_mm'],
        title='Sensibilidad diaria: mediana frente a media multiestación',
        labels={
            'precipitacion_mediana_estaciones_mm': 'Mediana (mm)',
            'precipitacion_media_estaciones_mm': 'Media (mm)',
        },
    )
    figura_sensibilidad.update_layout(height=620)
    return {
        'grafica_cobertura': figura_cobertura,
        'grafica_sensibilidad': figura_sensibilidad,
    }


def construir_reporte(resultado, manifest_entrada, inicio, fin, duracion):
    clasificacion = (
        resultado.cobertura_municipios['clasificacion_cobertura']
        .value_counts()
        .rename_axis('clasificacion_cobertura')
        .reset_index(name='municipios')
    )
    insuficiente = (
        resultado.cobertura_insuficiente
        .groupby(['departamento', 'codigo_municipio', 'municipio'], as_index=False)
        .agg(dias_cobertura_insuficiente=('fecha', 'size'))
        .sort_values('dias_cobertura_insuficiente', ascending=False)
    )
    return '\n'.join([
        '# Auditoría municipal de precipitación 2024-2025',
        '',
        f'- Contrato: `{AUDIT_VERSION}`',
        f'- Agregación auditada: `{manifest_entrada.get("aggregation_version")}`',
        f'- Commit ejecutor: `{detectar_commit(REPO_DIR)}`',
        f'- Commit de entrada: `{manifest_entrada.get("commit")}`',
        f'- Inicio: `{inicio.isoformat()}`',
        f'- Fin: `{fin.isoformat()}`',
        f'- Duración: `{formatear_duracion(duracion)}`',
        '',
        '> Auditoría de solo lectura. No elimina, imputa ni aprueba automáticamente la media, la mediana o un umbral de lluvia.',
        '',
        '## Métricas',
        '',
        tabla_markdown(pd.DataFrame([resultado.metricas])),
        '',
        '## Clasificación de cobertura municipal',
        '',
        tabla_markdown(clasificacion),
        '',
        '## Cobertura insuficiente por municipio',
        '',
        tabla_markdown(insuficiente),
        '',
        '## Sensibilidad de umbrales de lluvia',
        '',
        tabla_markdown(resultado.sensibilidad_umbrales_lluvia),
        '',
        '## Resumen de municipios multiestación',
        '',
        tabla_markdown(resultado.resumen_multiestacion),
        '',
        '## Mayor sensibilidad anual media-mediana',
        '',
        tabla_markdown(resultado.sensibilidad_media_mediana_anual, limite=25),
        '',
        '## Decisión',
        '',
        'La auditoría queda COMPLETA_CON_REVISION_PENDIENTE. Antes del paso 08 se deben revisar los municipios con mayor dispersión y definir cobertura mínima por periodo.',
        '',
    ])


def guardar_auditoria(resultado, manifest_entrada, archivos, reporte, figuras, inicio, fin, duracion):
    manifest_path = OUTPUT_DIR / NOMBRES_SALIDA['manifest']
    if manifest_path.exists() and not SOBRESCRIBIR_AUDITORIA:
        existente = leer_manifest_si_existe(manifest_path)
        if existente.get('estado') == 'COMPLETA_CON_REVISION_PENDIENTE':
            raise FileExistsError(f'La auditoría ya existe y no se sobrescribe: {OUTPUT_DIR}')

    tablas = {
        'cobertura_municipios': resultado.cobertura_municipios,
        'cobertura_periodos': resultado.cobertura_periodos,
        'cobertura_insuficiente': resultado.cobertura_insuficiente,
        'multiestacion_dias': resultado.multiestacion_dias,
        'resumen_multiestacion': resultado.resumen_multiestacion,
        'sensibilidad_anual': resultado.sensibilidad_media_mediana_anual,
        'sensibilidad_umbrales': resultado.sensibilidad_umbrales_lluvia,
    }
    salidas = {}
    for clave, tabla in tablas.items():
        ruta = OUTPUT_DIR / NOMBRES_SALIDA[clave]
        escribir_parquet_atomico(tabla, ruta, sobrescribir=SOBRESCRIBIR_AUDITORIA)
        salidas[clave] = {'ruta': str(ruta), 'filas': len(tabla)}

    reporte_path = OUTPUT_DIR / NOMBRES_SALIDA['reporte']
    escribir_texto_atomico(reporte, reporte_path, sobrescribir=SOBRESCRIBIR_AUDITORIA)
    for clave, figura in figuras.items():
        ruta = OUTPUT_DIR / NOMBRES_SALIDA[clave]
        escribir_texto_atomico(
            figura.to_html(full_html=True, include_plotlyjs='cdn'),
            ruta,
            sobrescribir=SOBRESCRIBIR_AUDITORIA,
        )
        salidas[clave] = {'ruta': str(ruta)}

    manifest = {
        'audit_version': AUDIT_VERSION,
        'estado': resultado.metricas['estado'],
        'commit': detectar_commit(REPO_DIR),
        'inicio': inicio.isoformat(),
        'fin': fin.isoformat(),
        'duracion_segundos': round(duracion, 2),
        'parametros': {'umbrales_lluvia_mm': UMBRALES_LLUVIA_MM},
        'entrada': {
            'ruta': str(INPUT_DIR),
            'commit': manifest_entrada.get('commit'),
            'aggregation_version': manifest_entrada.get('aggregation_version'),
            'particiones': len(archivos),
        },
        'metricas': resultado.metricas,
        'salidas': salidas,
        'reporte': str(reporte_path),
    }
    escribir_json_atomico(manifest, manifest_path, sobrescribir=True)
    print(f'Auditoría guardada en: {OUTPUT_DIR}')
    return manifest

## 5. Ejecución protegida

Con la bandera en `False` solo se muestra el plan. Con `True`, la auditoría lee las 48 particiones, genera evidencia y opcionalmente la guarda.

In [ ]:
resultado_auditoria_municipal = None

if not EJECUTAR_AUDITORIA_MUNICIPAL:
    print('Auditoría municipal desactivada. Revise el plan y active la bandera.')
else:
    inicio = ahora_proyecto()
    reloj = time.perf_counter()
    clima_municipal, manifest_entrada, archivos = cargar_clima_municipal()
    resultado_auditoria_municipal = auditar_precipitacion_municipal(
        clima_municipal,
        umbrales_lluvia_mm=UMBRALES_LLUVIA_MM,
    )
    figuras = construir_figuras(resultado_auditoria_municipal)
    fin = ahora_proyecto()
    duracion = time.perf_counter() - reloj
    reporte = construir_reporte(
        resultado_auditoria_municipal,
        manifest_entrada,
        inicio,
        fin,
        duracion,
    )

    display(Markdown('## Métricas de auditoría municipal'))
    display(pd.DataFrame([resultado_auditoria_municipal.metricas]))
    display(Markdown('## Cobertura por municipio'))
    display(resultado_auditoria_municipal.cobertura_municipios)
    display(Markdown('## Días con cobertura insuficiente'))
    display(resultado_auditoria_municipal.cobertura_insuficiente)
    display(Markdown('## Sensibilidad de umbrales de lluvia'))
    display(resultado_auditoria_municipal.sensibilidad_umbrales_lluvia)
    display(Markdown('## Resumen multiestación'))
    display(resultado_auditoria_municipal.resumen_multiestacion)
    for figura in figuras.values():
        figura.show()
    print(f'Duración: {formatear_duracion(duracion)}')

    if GUARDAR_RESULTADOS:
        guardar_auditoria(
            resultado_auditoria_municipal,
            manifest_entrada,
            archivos,
            reporte,
            figuras,
            inicio,
            fin,
            duracion,
        )
    else:
        print('Resultados no guardados porque GUARDAR_RESULTADOS=False.')

## 6. Cómo interpretar el resultado

- `COMPLETA_CON_REVISION_PENDIENTE` significa que la auditoría terminó, no que la regla municipal fue aprobada.
- Los acumulados de media y mediana se comparan usando exactamente los mismos días válidos; no extrapolan ausencias.
- Una diferencia grande puede representar heterogeneidad espacial real, estaciones no comparables o un problema de medición.
- Los municipios sin cobertura suficiente permanecen visibles y con `NaN`.
- El paso 08 solo se habilita después de documentar la decisión sobre dispersión y cobertura por periodo.

In [ ]:
# 7. Exploración opcional: estaciones del caso extremo de Aquitania
# Esta celda no modifica ni guarda artefactos del pipeline.
EJECUTAR_MAPA_ESTACIONES_AQUITANIA = False

if not EJECUTAR_MAPA_ESTACIONES_AQUITANIA:
    print('Mapa de Aquitania desactivado. Active EJECUTAR_MAPA_ESTACIONES_AQUITANIA.')
else:
    import html

    import folium
    import geopandas as gpd
    from shapely import wkb

    CODIGO_AQUITANIA = '15047'
    FECHA_CASO_AQUITANIA = pd.Timestamp('2024-07-05')
    ESTACIONES_CASO_AQUITANIA = ['0035167000', '0035167010', '0035195060']
    GEOGRAFIA_MAPA = 'estaciones_precipitacion_2024_2025_v3'
    CONSOLIDACION_MAPA = 'cierre_precipitacion_2024_2025_v2'

    geografia_dir = (
        PROCESSED_ROOT
        / 'geografia_curada'
        / f'canonica={slugificar(GEOGRAFIA_MAPA)}'
    )
    estaciones_path = geografia_dir / 'estaciones_municipio.parquet'
    poligonos_path = geografia_dir / 'divipola_municipios_geometria.parquet'
    clima_caso_path = (
        PROCESSED_ROOT
        / 'clima_diario_curado'
        / 'variable=precipitacion'
        / 'fuente=s54a-sgyg'
        / f'consolidacion={slugificar(CONSOLIDACION_MAPA)}'
        / 'departamento=BOYACÁ'
        / 'anio=2024'
        / 'mes=07'
        / 'observaciones_estacion_dia.parquet'
    )
    faltantes = [
        str(ruta)
        for ruta in (estaciones_path, poligonos_path, clima_caso_path)
        if not ruta.exists()
    ]
    if faltantes:
        raise FileNotFoundError(f'Faltan entradas para el mapa: {faltantes}')

    estaciones_geo = pd.read_parquet(estaciones_path)
    estaciones_geo['codigoestacion'] = estaciones_geo['codigoestacion'].astype('string')
    estaciones_geo['codigo_municipio_canonico'] = (
        estaciones_geo['codigo_municipio_canonico'].astype('string').str.zfill(5)
    )
    estaciones_caso = estaciones_geo.loc[
        estaciones_geo['codigoestacion'].isin(ESTACIONES_CASO_AQUITANIA)
        & estaciones_geo['codigo_municipio_canonico'].eq(CODIGO_AQUITANIA)
    ].copy()
    if set(estaciones_caso['codigoestacion']) != set(ESTACIONES_CASO_AQUITANIA):
        raise RuntimeError('No se encontraron las tres estaciones canónicas de Aquitania.')

    clima_caso = pd.read_parquet(clima_caso_path)
    clima_caso['fecha'] = pd.to_datetime(clima_caso['fecha'], errors='coerce')
    clima_caso['codigoestacion'] = clima_caso['codigoestacion'].astype('string')
    clima_caso = clima_caso.loc[
        clima_caso['fecha'].eq(FECHA_CASO_AQUITANIA)
        & clima_caso['codigoestacion'].isin(ESTACIONES_CASO_AQUITANIA),
        [
            'codigoestacion', 'precipitacion_diaria_mm',
            'calidad_dia', 'requiere_revision', 'motivos_revision',
        ],
    ]
    estaciones_caso = estaciones_caso.merge(
        clima_caso,
        on='codigoestacion',
        how='left',
        validate='one_to_one',
    )

    poligonos = pd.read_parquet(poligonos_path)
    poligonos['codigo_municipio_poligono'] = (
        poligonos['codigo_municipio_poligono'].astype('string').str.zfill(5)
    )
    poligono_aquitania = poligonos.loc[
        poligonos['codigo_municipio_poligono'].eq(CODIGO_AQUITANIA)
    ].copy()
    if len(poligono_aquitania) != 1:
        raise RuntimeError('No se encontró un único polígono para Aquitania.')
    if isinstance(poligono_aquitania['geometry'].iloc[0], (bytes, bytearray)):
        poligono_aquitania['geometry'] = poligono_aquitania['geometry'].map(wkb.loads)
    poligono_aquitania = gpd.GeoDataFrame(
        poligono_aquitania,
        geometry='geometry',
        crs='EPSG:4326',
    )

    centro = poligono_aquitania.geometry.iloc[0].representative_point()
    mapa_aquitania = folium.Map(
        location=[centro.y, centro.x],
        zoom_start=10,
        tiles='CartoDB positron',
    )
    folium.GeoJson(
        poligono_aquitania.__geo_interface__,
        name='Municipio de Aquitania',
        style_function=lambda _: {
            'color': '#1f5d50', 'weight': 3,
            'fillColor': '#b9d8c7', 'fillOpacity': 0.25,
        },
        tooltip='Aquitania (15047)',
    ).add_to(mapa_aquitania)

    for fila in estaciones_caso.itertuples(index=False):
        revision = bool(fila.requiere_revision) if pd.notna(fila.requiere_revision) else False
        color = '#d95f02' if revision else '#238443'
        lluvia = (
            'NaN'
            if pd.isna(fila.precipitacion_diaria_mm)
            else f'{float(fila.precipitacion_diaria_mm):.2f} mm'
        )
        contenido = (
            f'<b>{html.escape(str(fila.Nombre))}</b><br>'
            f'Estación: {html.escape(str(fila.codigoestacion))}<br>'
            f'Altitud IDEAM: {fila.altitud_ideam_m} m<br>'
            f'Precipitación {FECHA_CASO_AQUITANIA.date()}: {lluvia}<br>'
            f'Calidad: {html.escape(str(fila.calidad_dia))}<br>'
            f'Requiere revisión: {"Sí" if revision else "No"}'
        )
        folium.CircleMarker(
            location=[float(fila.LATITUD), float(fila.LONGITUD)],
            radius=8,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.9,
            tooltip=f'{fila.codigoestacion}: {lluvia}',
            popup=folium.Popup(contenido, max_width=360),
        ).add_to(mapa_aquitania)

    limites = poligono_aquitania.total_bounds
    mapa_aquitania.fit_bounds(
        [[limites[1], limites[0]], [limites[3], limites[2]]]
    )
    folium.LayerControl().add_to(mapa_aquitania)
    display(Markdown('### Tres estaciones del caso Aquitania, 5 de julio de 2024'))
    display(
        estaciones_caso[
            [
                'codigoestacion', 'Nombre', 'altitud_ideam_m',
                'LATITUD', 'LONGITUD', 'precipitacion_diaria_mm',
                'requiere_revision', 'calidad_dia',
            ]
        ].sort_values('codigoestacion')
    )
    display(mapa_aquitania)

In [ ]:
# 8. Exploración opcional: ventana temporal del caso Aquitania
# Combina serie temporal e histograma porque el histograma por sí solo no conserva el orden de las fechas.
EJECUTAR_HISTOGRAMA_AQUITANIA = False

if not EJECUTAR_HISTOGRAMA_AQUITANIA:
    print('Histograma de Aquitania desactivado. Active EJECUTAR_HISTOGRAMA_AQUITANIA.')
else:
    import matplotlib.pyplot as plt
    import numpy as np

    FECHA_EVENTO_AQUITANIA = pd.Timestamp('2024-07-05')
    DIAS_VENTANA_AQUITANIA = 15
    ESTACIONES_VENTANA_AQUITANIA = ['0035167000', '0035167010', '0035195060']
    CONSOLIDACION_VENTANA = 'cierre_precipitacion_2024_2025_v2'
    inicio_ventana = FECHA_EVENTO_AQUITANIA - pd.Timedelta(days=DIAS_VENTANA_AQUITANIA)
    fin_ventana = FECHA_EVENTO_AQUITANIA + pd.Timedelta(days=DIAS_VENTANA_AQUITANIA)
    meses_ventana = pd.period_range(inicio_ventana, fin_ventana, freq='M')

    archivos_ventana = [
        PROCESSED_ROOT
        / 'clima_diario_curado'
        / 'variable=precipitacion'
        / 'fuente=s54a-sgyg'
        / f'consolidacion={slugificar(CONSOLIDACION_VENTANA)}'
        / 'departamento=BOYACÁ'
        / f'anio={periodo.year}'
        / f'mes={periodo.month:02d}'
        / 'observaciones_estacion_dia.parquet'
        for periodo in meses_ventana
    ]
    faltantes = [str(ruta) for ruta in archivos_ventana if not ruta.exists()]
    if faltantes:
        raise FileNotFoundError(f'Faltan particiones para la ventana de Aquitania: {faltantes}')

    ventana = pd.concat(
        [
            pd.read_parquet(
                ruta,
                columns=[
                    'codigoestacion', 'fecha', 'precipitacion_diaria_mm',
                    'requiere_revision', 'calidad_dia',
                ],
            )
            for ruta in archivos_ventana
        ],
        ignore_index=True,
    )
    ventana['codigoestacion'] = ventana['codigoestacion'].astype('string')
    ventana['fecha'] = pd.to_datetime(ventana['fecha'], errors='coerce')
    ventana = ventana.loc[
        ventana['codigoestacion'].isin(ESTACIONES_VENTANA_AQUITANIA)
        & ventana['fecha'].between(inicio_ventana, fin_ventana)
    ].copy()
    if ventana.duplicated(['codigoestacion', 'fecha']).any():
        raise RuntimeError('La ventana contiene llaves estación-día repetidas.')
    conteos = ventana.groupby('codigoestacion')['fecha'].nunique()
    if set(conteos.index.astype(str)) != set(ESTACIONES_VENTANA_AQUITANIA):
        raise RuntimeError('No aparecen las tres estaciones en la ventana temporal.')

    resumen_ventana = (
        ventana.groupby('codigoestacion', as_index=False)
        .agg(
            dias=('fecha', 'nunique'),
            dias_con_valor=('precipitacion_diaria_mm', 'count'),
            media_mm=('precipitacion_diaria_mm', 'mean'),
            mediana_mm=('precipitacion_diaria_mm', 'median'),
            p95_mm=('precipitacion_diaria_mm', lambda serie: serie.quantile(0.95)),
            maximo_mm=('precipitacion_diaria_mm', 'max'),
            dias_revision=('requiere_revision', lambda serie: int(serie.fillna(False).sum())),
        )
    )
    evento = ventana.loc[
        ventana['fecha'].eq(FECHA_EVENTO_AQUITANIA),
        ['codigoestacion', 'precipitacion_diaria_mm'],
    ].rename(columns={'precipitacion_diaria_mm': 'valor_evento_mm'})
    resumen_ventana = resumen_ventana.merge(
        evento,
        on='codigoestacion',
        how='left',
        validate='one_to_one',
    )

    colores = {
        '0035167000': '#238443',
        '0035167010': '#d95f02',
        '0035195060': '#2b6cb0',
    }
    figura, (eje_serie, eje_histograma) = plt.subplots(
        2,
        1,
        figsize=(14, 10),
        gridspec_kw={'height_ratios': [1.35, 1]},
    )
    for codigo in ESTACIONES_VENTANA_AQUITANIA:
        serie = ventana.loc[ventana['codigoestacion'].eq(codigo)].sort_values('fecha')
        eje_serie.plot(
            serie['fecha'],
            serie['precipitacion_diaria_mm'],
            marker='o',
            markersize=4,
            linewidth=1.8,
            color=colores[codigo],
            label=codigo,
        )
    maximo_entre_estaciones = (
        ventana.groupby('fecha', as_index=False)['precipitacion_diaria_mm'].max()
    )
    eje_serie.plot(
        maximo_entre_estaciones['fecha'],
        maximo_entre_estaciones['precipitacion_diaria_mm'],
        color='#222222',
        linestyle='--',
        linewidth=1.5,
        label='Máximo diario entre las 3 estaciones',
    )
    eje_serie.axvline(
        FECHA_EVENTO_AQUITANIA,
        color='#9b2226',
        linestyle=':',
        linewidth=2,
        label='5 de julio de 2024',
    )
    eje_serie.set_title('Aquitania: precipitación diaria, 15 días antes y después')
    eje_serie.set_ylabel('Precipitación diaria (mm)')
    eje_serie.grid(alpha=0.25)
    eje_serie.legend(ncol=2)

    maximo_global = float(ventana['precipitacion_diaria_mm'].max())
    bins_comunes = np.linspace(0, maximo_global * 1.05, 16)
    for codigo in ESTACIONES_VENTANA_AQUITANIA:
        valores = ventana.loc[
            ventana['codigoestacion'].eq(codigo),
            'precipitacion_diaria_mm',
        ].dropna()
        eje_histograma.hist(
            valores,
            bins=bins_comunes,
            alpha=0.48,
            color=colores[codigo],
            edgecolor='white',
            label=codigo,
        )
    eje_histograma.axvline(
        261.3,
        color='#9b2226',
        linestyle=':',
        linewidth=2,
        label='Pico 261,3 mm',
    )
    eje_histograma.set_title('Distribución de los acumulados diarios en la ventana')
    eje_histograma.set_xlabel('Precipitación diaria (mm)')
    eje_histograma.set_ylabel('Número de días')
    eje_histograma.grid(axis='y', alpha=0.25)
    eje_histograma.legend(ncol=2)
    figura.tight_layout()

    display(Markdown('### Ventana temporal de las tres estaciones de Aquitania'))
    print(
        'Nota: precipitacion_diaria_mm es el acumulado diario consolidado; '
        'no representa un máximo subdiario.'
    )
    display(resumen_ventana.round(2))
    plt.show()